# 모델 성능 평가: YOLO26 디텍터 + ResNet18 분류기 2종

`train_models.ipynb`가 만든 세 가중치(`weights/yolo26_detector_best.pt`,
`weights/resnet18_number_best.pt`, `weights/resnet18_color_best.pt`)를 각각 평가합니다.

**테스트 데이터**: 별도로 샘플을 추리지 않고, 각 데이터셋의 `test` 폴더에 있는 이미지를
**전부** 그대로 사용합니다.

| 모델 | 테스트 데이터 | 지표 |
|---|---|---|
| YOLO26 | `rummikub_data_3/images(labels)/test` 전체 | precision/recall/mAP50/mAP50-95 (+ 클래스별) |
| ResNet18 숫자 | `classifier_number/test` 전체 | 정확도 + 클래스별 precision/recall |
| ResNet18 색상 | `classifier_color/test` 전체 | 정확도 + 클래스별 precision/recall |

이전 노트북들과 동일하게 cwd가 `.../boardgame-ai/rummikub_0730`라고 가정한 상대경로만 쓰고,
로컬 프로젝트 경로 의존은 없습니다. 이 노트북은 학습을 하지 않고 저장된 가중치만 불러와서
평가만 수행합니다(커널이 따로 떠도 동작하도록 자체적으로 다시 설정합니다).

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch

WEIGHTS_DIR = Path("weights")
RUMMIKUB_DATA_3 = Path("rummikub_data_3")
CLASSIFIER_NUMBER = Path("classifier_number")
CLASSIFIER_COLOR = Path("classifier_color")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

for name, path in {
    "YOLO26 detector": WEIGHTS_DIR / "yolo26_detector_best.pt",
    "ResNet18 number": WEIGHTS_DIR / "resnet18_number_best.pt",
    "ResNet18 color": WEIGHTS_DIR / "resnet18_color_best.pt",
}.items():
    print(f"{name}: {path.resolve()} -> {'존재' if path.is_file() else '없음!'}")

# ---- rummikub/classes.py의 53클래스 정의를 그대로 옮겨온 것 (외부 의존성 없음) ----
COLORS = ("black", "blue", "orange", "red")


def _raw_label(canonical_name: str) -> str:
    if canonical_name == "joker":
        return "Joker"
    color, number = canonical_name.rsplit("_", 1)
    return f"{color.capitalize()}{number}"


_NUMERIC_ORDER = [f"{color}_{number}" for color in COLORS for number in range(1, 14)] + ["joker"]
CLASS_NAMES = sorted(_NUMERIC_ORDER, key=_raw_label)
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def canonicalize_external_name(name: str) -> str:
    compact = "".join(ch for ch in name.lower() if ch.isalnum())
    if compact == "joker":
        return "joker"
    for color in COLORS:
        suffix = compact[len(color):] if compact.startswith(color) else ""
        if suffix.isdigit():
            candidate = f"{color}_{int(suffix)}"
            if candidate in CLASS_TO_ID:
                return candidate
    raise ValueError(f"Unknown external Rummikub class: {name}")

## 1. YOLO26 디텍터 평가 (`rummikub_data_3/images(labels)/test` 전체)

학습에 쓰지 않은 test 폴더 전체를 그대로 Ultralytics `model.val(split="test")`로 평가합니다.
train/val 항목은 yaml 스키마상 필요해서 test와 동일한 경로로 채워두지만 실제 평가에는 쓰이지 않습니다.

In [ ]:
test_image_dir = RUMMIKUB_DATA_3 / "images" / "test"
assert test_image_dir.is_dir(), f"test 이미지 폴더가 없습니다: {test_image_dir.resolve()}"
n_test_images = len(list(test_image_dir.glob("*")))
print(f"test 이미지 {n_test_images}장: {test_image_dir.resolve()}")

names_yaml = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))
eval_yaml_text = (
    f"train: {test_image_dir.resolve()}\n"  # 평가만 하므로 실제로 쓰이지 않음
    f"val: {test_image_dir.resolve()}\n"  # 평가만 하므로 실제로 쓰이지 않음
    f"test: {test_image_dir.resolve()}\n"
    "\n"
    "names:\n"
    f"{names_yaml}\n"
)
eval_dir = Path("yolo_eval")
eval_dir.mkdir(exist_ok=True)
eval_yaml_path = eval_dir / "dataset.yaml"
eval_yaml_path.write_text(eval_yaml_text, encoding="utf-8")
print(eval_yaml_text)

In [ ]:
from ultralytics import YOLO

yolo_weights_path = WEIGHTS_DIR / "yolo26_detector_best.pt"
yolo_model = YOLO(str(yolo_weights_path))

model_names = [yolo_model.names[i] for i in sorted(yolo_model.names)]
try:
    canonical_from_model = [canonicalize_external_name(n) for n in model_names]
except ValueError:
    canonical_from_model = None
if canonical_from_model != CLASS_NAMES:
    print(
        "[경고] 이 가중치의 클래스 순서가 CLASS_NAMES와 다릅니다. "
        "precision/recall/mAP가 클래스별로 잘못 매칭되어 계산될 수 있습니다.\n"
        f"  모델 클래스 순서(0~4): {model_names[:5]}\n"
        f"  CLASS_NAMES 순서(0~4): {CLASS_NAMES[:5]}"
    )

yolo_metrics = yolo_model.val(
    data=str(eval_yaml_path.resolve()),
    split="test",
    imgsz=960,
    conf=0.001,  # mAP는 전체 후보 박스에 대해 적분하므로 낮은 임계값 사용 (추론 임계값 아님)
    iou=0.6,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/detect",
    name="rummikub_yolo26_eval",
    plots=True,
)

print(f"\n[test] {yolo_weights_path}")
print(f"  Precision (mean): {yolo_metrics.box.mp:.3f}")
print(f"  Recall (mean):    {yolo_metrics.box.mr:.3f}")
print(f"  mAP50:            {yolo_metrics.box.map50:.3f}")
print(f"  mAP50-95:         {yolo_metrics.box.map:.3f}")

print("\nclass별 mAP50-95 (0.000 = 이 split에 예측/정답이 없음):")
for class_id, class_map in enumerate(yolo_metrics.box.maps):
    name = CLASS_NAMES[class_id] if class_id < len(CLASS_NAMES) else f"class_{class_id}"
    print(f"  {name:12s} {class_map:.3f}")

print("\nPR curve / confusion matrix 저장 위치: runs/detect/rummikub_yolo26_eval/")

## 2. ResNet18 분류기 공통 평가 함수
`classifier_number/test`, `classifier_color/test`에 있는 이미지를 전부 그대로 평가합니다.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

IMG_SIZE = 96
BATCH_SIZE = 64
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def build_resnet_classifier(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


def evaluate_classifier(weights_path: Path, test_dir: Path, label: str) -> dict:
    assert test_dir.is_dir(), f"[{label}] test 폴더가 없습니다: {test_dir.resolve()}"
    ckpt = torch.load(weights_path, map_location=DEVICE)
    classes = ckpt["classes"]

    test_ds = datasets.ImageFolder(test_dir, transform=eval_tf)
    assert test_ds.classes == classes, (
        f"[{label}] test 폴더의 클래스 목록이 학습 당시 저장된 클래스 목록과 다릅니다.\n"
        f"  test 폴더: {test_ds.classes}\n  저장된 순서: {classes}"
    )
    print(f"[{label}] test 이미지 {len(test_ds)}장, 클래스 {len(classes)}개: {classes}")

    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model = build_resnet_classifier(len(classes))
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    n_classes = len(classes)
    confusion = np.zeros((n_classes, n_classes), dtype=int)

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            preds = model(images).argmax(dim=1).cpu().numpy()
            for true_idx, pred_idx in zip(labels.numpy(), preds):
                confusion[true_idx, pred_idx] += 1

    total = confusion.sum()
    accuracy = confusion.trace() / total if total else 0.0

    print(f"\n[{label}] 전체 정확도: {accuracy:.4f} ({confusion.trace()}/{total})")
    print(f"{'class':<10}{'support':>10}{'precision':>12}{'recall':>10}")
    for i, class_name in enumerate(classes):
        support = confusion[i].sum()
        tp = confusion[i, i]
        col_sum = confusion[:, i].sum()
        precision = tp / col_sum if col_sum else float("nan")
        recall = tp / support if support else float("nan")
        print(f"{class_name:<10}{support:>10}{precision:>12.3f}{recall:>10.3f}")

    return {"label": label, "accuracy": accuracy, "confusion": confusion, "classes": classes}

## 3. 숫자 분류기 평가 (`classifier_number/test` 전체)

In [ ]:
number_report = evaluate_classifier(
    WEIGHTS_DIR / "resnet18_number_best.pt", CLASSIFIER_NUMBER / "test", "number"
)

## 4. 색상 분류기 평가 (`classifier_color/test` 전체)

In [ ]:
color_report = evaluate_classifier(
    WEIGHTS_DIR / "resnet18_color_best.pt", CLASSIFIER_COLOR / "test", "color"
)

## 5. 요약

In [ ]:
print("YOLO26 detector  mAP50={:.3f}  mAP50-95={:.3f}  P={:.3f}  R={:.3f}".format(
    yolo_metrics.box.map50, yolo_metrics.box.map, yolo_metrics.box.mp, yolo_metrics.box.mr
))
print(f"ResNet18 number  accuracy={number_report['accuracy']:.4f}")
print(f"ResNet18 color   accuracy={color_report['accuracy']:.4f}")